In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from functions import create_DF, nfinca, welcome, goodbye
import os

In [2]:
from functions import *

In [3]:
#cargar libros de excel con datos de curvas
file = 'C:/Users/Estadistica/Desktop/py/curvas/files/bd.xlsx'
sh_file = "bd_python"
col_file = ['nvariedad','edad','pico','ciclo','fm_real','fm_proy','fm_aj','fm_suave']
converts={}
file_fus = 'C:/Users/Estadistica/Desktop/py/curvas/files/historia_fusarium.xlsx'
sh_fus = "Hoja1"
col_fus = ['fusarium','variedad','pico','finca','edad']
file_ciclo = 'C:/Users/Estadistica/Desktop/py/curvas/files/historia_fusarium.xlsx'
sh_ciclo = "ciclos"

In [4]:
#-----------------------------------------
df = create_DF(file,sh_file,col_file,converts)
#-----------------------------------------
#cargar datos de fusarium
df_fus = create_DF(file_fus,sh_fus,col_fus,converts)
#-----------------------------------------
#-----------------------------------------
#cargar datos de fusarium
#df_ciclos = create_DF(file_ciclo,sh_ciclo)
#-----------------------------------------
df['finca'] = 0

In [5]:
#crear grafico
def graficar(vari,finca,df,df_fus,df_ciclos,option):
    #maxi
    maxi = 0
    #------------------------------------
    df = df[ df.nvariedad==vari ]
    df = df[ df.finca==finca ]
    #traer datos de fusarium
    df_fus = df_fus[df_fus.variedad==vari]
    df_fus = df_fus[df_fus.finca==finca]
    #datos del ciclo
    df_ciclos = df_ciclos[df_ciclos.variedad==vari]
    # definir jhvariables x,y
    xh = np.array(df_fus['edad'])
    yh = np.array(df_fus['fusarium'])
    xh = xh.reshape(-1, 1)
    yh = yh.reshape(-1, 1)
    #calcular variables ciclo
    xc = np.array(df_ciclos['ciclo'])
    xc = xc.reshape(-1,1)
    #calcular variables ciclo ajustado
    ciclo_aj = np.array(np.mean(df['ciclo']))
    ciclo_aj = ciclo_aj.reshape(-1,1)
    #ordenar por edad
    df.sort_values(by=['edad'])
    maxr = max(df.fm_real)
    maxp = max(df.fm_proy)
    maxa = max(df.fm_suave)
    maxi = max(maxr,maxp,maxa)
    #calcular subototales para picos
    df_p1 = df[df.pico==1]
    df_p2 = df[df.pico==2]
    df_p3 = df[df.pico==3]
    r_p1 = np.round(df_p1.fm_real.sum(),2)
    r_p2 = np.round(df_p2.fm_real.sum(),2)
    r_p3 = np.round(df_p3.fm_real.sum(),2)
    t_p1 = np.round(df_p1.fm_proy.sum(),2)
    t_p2 = np.round(df_p2.fm_proy.sum(),2)
    t_p3 = np.round(df_p3.fm_proy.sum(),2)
    a_p1 = np.round(df_p1.fm_suave.sum(),2)
    a_p2 = np.round(df_p2.fm_suave.sum(),2)
    a_p3 = np.round(df_p3.fm_suave.sum(),2)
    total_r = np.round(np.sum(df.fm_real),2)
    total_t = np.round(np.sum(df.fm_proy),2)
    total_a = np.round(np.sum(df.fm_suave),2)
    #Fusarium Totales Para Graficar
    p1_fo = df_fus[df_fus.edad==27]
    p1_fo = np.round(np.average(p1_fo.fusarium * 100),2)
    p2_fo = df_fus[df_fus.edad==61]
    p2_fo = np.round(np.average(p2_fo.fusarium * 100),2)
    p3_fo = df_fus[df_fus.edad==100]
    p3_fo = np.round(np.average(p3_fo.fusarium * 100),2)
    #--------------------------------------#
    #print(f'1 {p1_fo} 2 {p2_fo}')
    #-----------GRAFICAR-------------------#
    fig, ax = plt.subplots()
    #----------define ejes---------------------#
    x_tick = np.arange(18,103,2)
    y_tick = np.arange(0,maxi + 0.2,0.05)
    #---------configura ejes, limites----------#
    ax.set_xticks(x_tick)
    ax.set_yticks(y_tick)
    ax.set_ylim(0)
    fig.suptitle(str(nfinca(finca)))
    ax.set_title( vari )
    fig.set_figheight(7)
    fig.set_figwidth(12)
    #----------------------------------------------------------
    plt.scatter(xh,yh,label='Dato historico fusarium',marker='x', c='black', s=100) #Datos historicos
    #plt.axvline(x=xc - 4, ymin=0, ymax=maxi, color='r', label='Inicia Ciclo', alpha=0.7)
    plt.axvline(x=xc, ymin=0, ymax=maxi, color='r', ls='--', label='Ciclo 2024', alpha=0.7)
    #---------------------------------------------------------#
    plt.axvline(x=ciclo_aj, ymin=0, ymax=maxi, color='m', ls='--', label='Ciclo Sugerido', alpha=0.7)
    #----------------------ingresar notas----------------------
    median = int(np.max(df.edad)/2)
    if median<18:
        median=18
    ax.text(median,maxi-0.05,'Notas',fontsize=16)
    ax.text(median,maxi-0.10,'FM Total: ' + str(total_r) + ' Real | ' + str(total_t) + ' Proy | ' + str(total_a) + ' Aj ',fontsize=14)
    ax.text(median,maxi-0.14,'FM 1P: ' + str(r_p1) + ' Real | ' + str(t_p1) + ' Proy | ' + str(a_p1) + ' Aj',fontsize=14)
    ax.text(median,maxi-0.18,'FM 2P: ' + str(r_p2) + ' Real | ' + str(t_p2) + ' Proy | ' + str(a_p2) + ' Aj',fontsize=14)
    ax.text(median,maxi-0.22,'FM 3P: ' + str(r_p3) + ' Real | ' + str(t_p3) + ' Proy | ' + str(a_p3) + ' Aj',fontsize=14)
    ax.text(median,maxi-0.26,'FO | 1P: ' + str(p1_fo) + '% | 2P: ' + str(p2_fo) + '% | 3P: ' + str(p3_fo) + '%',fontsize=12)
    #msg = input("Ingrese comentario: ")

    #------------configurar grafico-----------------------------
    ax.grid(linestyle='dotted', color='b',linewidth=0.5)
    ax.plot(df.edad,df.fm_real,label='FM real')
    ax.plot(df.edad,df.fm_proy,label='FM proy')
    ax.plot(df.edad,df.fm_suave,label='FM ajust')
    ax.set_xlabel('Edad')
    ax.set_ylabel('Flor Mata')
    ax.legend()
    fig.tight_layout()
    if int(option)==0:
        route_delete = "C:/Users/ESTADISTICA/Desktop/py/curvas"
        str_file = './img/final_curvas/' + str(vari) + '_' + str(finca) + '.png'
        if os.path.isfile(str_file):
            try:
                os.rm(route_delete + str_file)   # Opt.: os.system("rm "+strFile)
            except:
                print(f'No existia grafica de: {vari} en la finca {finca}')
        plt.savefig(str_file, transparent=True, bbox_inches='tight')
    else:
        plt.show()
    plt.close('all')

In [7]:
if __name__ == '__main__':
    while True:
        try:
            welcome()
            option = int(input("Seleccione opciones:\n 0: Graficar todo \n 1: Graficar manualmente \n 2: Para salir \n Escriba: "))
            if option==2:
                break
            break
        except ValueError:
            print("¡Debe ingresar un número de 0 a 2!")


    if option==0:
        #graficar todo
        varieties = df.nvariedad.unique().tolist()
        farms = df.finca.unique().tolist()
        for finca in farms:
            for vari in varieties:
                try:
                    graficar(vari,finca,df,df_fus,df_ciclos,option)
                except:
                    print(f'La variedad {vari} no existe en la finca {finca}')
    elif option==1:

        while True:
            try:
                vari = input("Ingrese el nombre de la variedad a graficar: ").upper().rstrip()
                if vari=='EXIT':
                    break
                else:
                    finca = int(input("Ingrese: 0 - Total finca, 1 - Palmas, 2 - Palermo: "))
                    graficar(vari,finca,df,df_fus,df_ciclos,option)
                    break
            except:
                print("Algo está mal, pruebe de nuevo. Si desea salir escriba 'exit' ")
    elif option==2:
        goodbye()
    else:
        print("Seleccione entre 0 y 2, como indica el menú.")

****************************************
Bienvenido 
Generador de Curvas - Inverpalmas
****************************************
Seleccione opciones:
 0: Graficar todo 
 1: Graficar manualmente 
 2: Para salir 
 Escriba: 0
